# Thử nghiệm ASR end-to-end cho Smart Bedroom (Colab)

Mục tiêu: đo **WER**, **độ chính xác lệnh** và **thời gian xử lý** của Whisper và PhoWhisper trên chính dữ liệu thu của nhóm, theo luồng
`giọng nói → văn bản → ánh xạ lệnh`. Kết quả dùng cho phần khảo sát và so sánh mô hình (mục 1.2–1.5 và 3.4 của phiếu chấm).

**Chuẩn bị**
1. `Runtime → Change runtime type → T4 GPU` (có GPU thì mới đo được thời gian GPU; không có vẫn chạy được, chậm hơn).
2. Nén thư mục `dataset/` của dự án thành `dataset.zip` (bên trong phải là `dataset/<lệnh>/<người>_<số>.wav`).
3. Chạy lần lượt từng ô.

**Lưu ý khi đọc kết quả**
- Whisper luôn xử lý cửa sổ 30 s (đệm thêm im lặng) nên thời gian gần như không phụ thuộc độ dài file 2 s.
- Thời gian trên Colab không bằng thời gian trên laptop demo. Ô cuối có đo cả CPU để tham khảo, nhưng Colab chỉ có ít nhân CPU.
- Lớp `noise` không có văn bản chuẩn: dùng để xem ASR có "nghe ra chữ" từ tiếng nền hay không (ảo giác), không tính vào WER.

In [ ]:
!pip -q install -U transformers jiwer soundfile librosa

## 1. Nạp dữ liệu

In [ ]:
import os, zipfile
from google.colab import files

up = files.upload()                       # chọn dataset.zip
for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('/content')

# Hoặc dùng Google Drive thay cho upload:
# from google.colab import drive; drive.mount('/content/drive'); DATA = '/content/drive/MyDrive/smart-bedroom/dataset'
DATA = '/content/dataset'
assert os.path.isdir(DATA), "Không thấy /content/dataset – zip phải chứa thư mục dataset/<lệnh>/*.wav"
print(sorted(os.listdir(DATA)))

## 2. Cấu hình

In [ ]:
MODELS = [
    "openai/whisper-base",
    "openai/whisper-small",
    "vinai/PhoWhisper-base",
    "vinai/PhoWhisper-small",
    # "openai/whisper-medium", "vinai/PhoWhisper-medium",   # chính xác hơn nhưng chậm, nặng
]
MAX_PER_CMD   = 30      # tối đa số file mỗi lệnh dùng để đánh giá (giảm nếu chạy lâu)
CPU_PER_CMD   = 3       # số file mỗi lệnh dùng để đo thời gian CPU (khi có GPU)
FUZZY_THRESHOLD = 0.75  # ngưỡng khớp mờ khi ánh xạ văn bản -> lệnh

# Câu chuẩn (reference) cho từng lệnh. Lớp noise không có câu chuẩn.
REFS = {
    "bat_dieu_hoa": "bật điều hòa",
    "tat_dieu_hoa": "tắt điều hòa",
    "bat_den_tran": "bật đèn trần",
    "tat_den_tran": "tắt đèn trần",
    "mo_rem":       "mở rèm",
    "bat_tivi":     "bật tivi",
}

## 3. Hàm hỗ trợ: chuẩn hóa văn bản và ánh xạ văn bản → lệnh

- `norm`: chữ thường, bỏ dấu câu, đưa kiểu đặt dấu thanh về một dạng (hòa/hoà là một).
- `text_to_command`: so khớp mờ (không dấu) với 6 câu lệnh; dưới ngưỡng thì coi là "không phải lệnh".

In [ ]:
import re, unicodedata, difflib

TONES = ['\u0300', '\u0301', '\u0309', '\u0303', '\u0323']
OLD2NEW = {}   # kiểu cũ 'hoà' -> kiểu mới 'hòa'
for t in TONES:
    for a, b in [('o', 'a'), ('o', 'e')]:
        OLD2NEW[a + unicodedata.normalize('NFC', b + t)] = unicodedata.normalize('NFC', a + t) + b

def norm(s):
    s = unicodedata.normalize('NFC', s.lower())
    for old, new in OLD2NEW.items():
        s = s.replace(old, new)
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_marks(s):
    s = unicodedata.normalize('NFD', s.replace('đ', 'd').replace('Đ', 'D'))
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')

PHRASES = {cmd: strip_marks(norm(txt)) for cmd, txt in REFS.items()}

def text_to_command(text):
    t = strip_marks(norm(text))
    if not t:
        return None, 0.0
    best_cmd, best = None, 0.0
    for cmd, ph in PHRASES.items():
        r = difflib.SequenceMatcher(None, t, ph).ratio()
        if ph in t:
            r = max(r, 0.95)
        if r > best:
            best_cmd, best = cmd, r
    return (best_cmd, best) if best >= FUZZY_THRESHOLD else (None, best)

In [ ]:
import random
from pathlib import Path
import numpy as np, pandas as pd, soundfile as sf, librosa

rng = random.Random(0)
rows = []
for cdir in sorted(Path(DATA).iterdir()):
    if not cdir.is_dir():
        continue
    fs = sorted(cdir.glob('*.wav')); rng.shuffle(fs)
    for f in fs[:MAX_PER_CMD]:
        y, sr = sf.read(f, dtype='float32', always_2d=True); y = y.mean(axis=1)
        if sr != 16000:
            y = librosa.resample(y, orig_sr=sr, target_sr=16000)
        rows.append(dict(path=str(f), cmd=cdir.name, speaker=f.stem.split('_')[0],
                         ref=REFS.get(cdir.name, ''), audio=y))
data = pd.DataFrame(rows)
print(data.groupby('cmd').size().to_string())
print("Người nói:", sorted(data.speaker.unique()))

## 4. Chạy ASR và đo thời gian

In [ ]:
import gc, time, torch
from tqdm.auto import tqdm
from transformers import pipeline

HAS_GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "không có (Runtime → Change runtime type → T4 GPU)")
sync = (lambda: torch.cuda.synchronize()) if HAS_GPU else (lambda: None)
GEN = {"language": "vi", "task": "transcribe"}

def run_model(name, device, df, warmup=3):
    t0 = time.perf_counter()
    asr = pipeline("automatic-speech-recognition", model=name,
                   device=0 if device == "cuda" else -1,
                   torch_dtype=torch.float16 if device == "cuda" else torch.float32)
    sync(); load_s = time.perf_counter() - t0
    infer = lambda y: asr({"raw": y, "sampling_rate": 16000}, generate_kwargs=GEN)["text"].strip()
    for _ in range(warmup):                      # khởi động, không tính vào thời gian
        infer(df.audio.iloc[0])
    out = []
    for r in tqdm(df.itertuples(), total=len(df), desc=f"{name} [{device}]"):
        sync(); t = time.perf_counter(); hyp = infer(r.audio); sync()
        asr_ms = (time.perf_counter() - t) * 1000
        t = time.perf_counter(); pred, score = text_to_command(hyp)
        map_ms = (time.perf_counter() - t) * 1000
        out.append(dict(model=name, device=device, path=r.path, cmd=r.cmd, speaker=r.speaker, ref=r.ref,
                        hyp=hyp, pred=pred, score=score, asr_ms=asr_ms, map_ms=map_ms, load_s=load_s))
    del asr; gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()
    return pd.DataFrame(out)

cpu_data = data.groupby('cmd').head(CPU_PER_CMD) if HAS_GPU else data
results = []
for m in MODELS:
    if HAS_GPU:
        results.append(run_model(m, "cuda", data))
    results.append(run_model(m, "cpu", cpu_data))
df = pd.concat(results, ignore_index=True)

## 5. Bảng kết quả tổng hợp

In [ ]:
import jiwer

def summarize(df):
    rows = []
    for (model, device), g in df.groupby(['model', 'device'], sort=False):
        c, n = g[g.cmd != 'noise'], g[g.cmd == 'noise']
        refs = [norm(r) for r in c.ref]
        hyps = [norm(h) or "_" for h in c.hyp]        # '_' thay chuỗi rỗng (jiwer không nhận)
        rows.append({
            'model': model, 'thiết bị': device, 'số mẫu lệnh': len(c),
            'WER (có dấu)': jiwer.wer(refs, hyps),
            'WER (không dấu)': jiwer.wer([strip_marks(r) for r in refs], [strip_marks(h) for h in hyps]),
            'Acc lệnh': (c.pred == c.cmd).mean(),
            'Nhầm bật/tắt': ((c.pred.notna()) & (c.pred != c.cmd)).mean(),
            'Không nhận ra': c.pred.isna().mean(),
            'Noise bị nhận thành lệnh': n.pred.notna().mean() if len(n) else np.nan,
            'Noise có ra chữ': (n.hyp.map(norm) != '').mean() if len(n) else np.nan,
            'ASR TB (ms)': g.asr_ms.mean(), 'ASR trung vị (ms)': g.asr_ms.median(),
            'ASR p95 (ms)': g.asr_ms.quantile(0.95), 'Ánh xạ lệnh (ms)': g.map_ms.mean(),
            'Thời gian nạp (s)': g.load_s.iloc[0],
        })
    return pd.DataFrame(rows)

summary = summarize(df)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
summary.round(3)

**Cách đọc cột**
- *WER*: tỉ lệ lỗi từ giữa câu nhận dạng và câu chuẩn (thấp hơn là tốt). Bản "không dấu" bỏ qua lỗi dấu thanh để thấy mô hình có nghe đúng âm hay không.
- *Acc lệnh*: sau khi ánh xạ văn bản thành lệnh có đúng lệnh không (đây là chỉ số end-to-end cho hệ thống).
- *Nhầm bật/tắt*: nhận ra một lệnh nhưng **sai** lệnh; *Không nhận ra*: văn bản không khớp lệnh nào.
- *Noise bị nhận thành lệnh*: tỉ lệ tiếng nền bị kích hoạt thành lệnh (càng thấp càng an toàn).

## 6. Xem chi tiết

In [ ]:
gpu = df[df.device == ('cuda' if HAS_GPU else 'cpu')]
for m, g in gpu.groupby('model', sort=False):
    print(f"\n=== {m} ===")
    print(g[g.cmd != 'noise'][['cmd', 'hyp', 'pred']].head(8).to_string(index=False))

In [ ]:
# Tiếng nền bị "nghe" thành chữ gì? (ảo giác của Whisper thường lộ ra ở đây)
for m, g in gpu.groupby('model', sort=False):
    nz = g[g.cmd == 'noise'].hyp.map(lambda s: s.strip() or '(rỗng)').value_counts().head(5)
    print(f"\n=== {m} ===\n{nz.to_string()}")

In [ ]:
# Ma trận nhầm lẫn lệnh (end-to-end) của từng mô hình
for m, g in gpu.groupby('model', sort=False):
    c = g[g.cmd != 'noise']
    print(f"\n=== {m} ===")
    print(pd.crosstab(c.cmd, c.pred.fillna('(không nhận ra)')).to_string())

In [ ]:
# WER theo người nói (giọng người nào khó hơn?)
c = gpu[gpu.cmd != 'noise']
tab = {}
for (m, s), g in c.groupby(['model', 'speaker'], sort=False):
    tab.setdefault(m, {})[s] = jiwer.wer([norm(r) for r in g.ref], [norm(h) or '_' for h in g.hyp])
pd.DataFrame(tab).T.round(3)

## 7. Biểu đồ và lưu kết quả

In [ ]:
import matplotlib.pyplot as plt
s = summary[summary['thiết bị'] == ('cuda' if HAS_GPU else 'cpu')]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].barh(s.model, s['WER (có dấu)']); ax[0].set_xlabel('WER (có dấu) – thấp hơn là tốt'); ax[0].set_title('Sai số từ')
ax[1].scatter(s['ASR TB (ms)'], s['Acc lệnh'], s=70)
for _, r in s.iterrows():
    ax[1].annotate(r['model'].split('/')[-1], (r['ASR TB (ms)'], r['Acc lệnh']), textcoords='offset points', xytext=(5, 5), fontsize=8)
ax[1].set_xlabel('Thời gian ASR trung bình (ms)'); ax[1].set_ylabel('Acc lệnh end-to-end'); ax[1].set_title('Độ chính xác so với độ trễ')
plt.tight_layout(); plt.savefig('asr_comparison.png', dpi=150); plt.show()

In [ ]:
df.drop(columns=[]).to_csv('asr_results.csv', index=False)
summary.to_csv('asr_summary.csv', index=False)
for f in ['asr_summary.csv', 'asr_results.csv', 'asr_comparison.png']:
    files.download(f)

## Gợi ý đưa vào báo cáo

- Trình bày bảng `summary` (WER, Acc lệnh, ASR TB, thời gian nạp) và biểu đồ độ chính xác so với độ trễ.
- So sánh với bộ phân loại tự huấn luyện (SVM/Random Forest trên MFCC): ASR có thể nhận câu tự do nhưng nặng và chậm hơn; bộ phân loại nhỏ nhanh, chạy offline nhưng chỉ nhận bộ lệnh cố định. Đây là cơ sở cho mục 1.4 và 1.5.
- Nêu rõ số mẫu và số người nói: kết quả trên tập nhỏ chỉ mang tính tham khảo.
- Xem mục "Noise có ra chữ" để bàn về vấn đề ảo giác của ASR khi không có tiếng nói.